# 02 - Cleaning, Buang Netral, Embedding (Ollama), Clustering per Label

In [9]:
import re
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from sklearn.cluster import AgglomerativeClustering

BASE_DIR = Path(r"C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data")
LABELED_DIR = BASE_DIR / "labeled"
PRAPROCESSING_DIR = BASE_DIR / "praprocessing"

# ===== UBAH DI SINI =====
INPUT_FILENAME = "labels_shafwa_hans_agreement.csv"
DROP_LABELS = ["netral"]                  # label yang dibuang
OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"                    # ollama pull bge-m3
SIM_THRESHOLD = 0.70                      # komentar dgn cosine similarity >= ini dianggap 1 konteks
# ========================

INPUT_PATH = LABELED_DIR / INPUT_FILENAME
OUTPUT_DIR = PRAPROCESSING_DIR / INPUT_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input :", INPUT_PATH)
print("Output:", OUTPUT_DIR)

Input : C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\labeled\labels_shafwa_hans_agreement.csv
Output: C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement


In [10]:
# ===== CLEANING + BUANG NETRAL =====
df = pd.read_csv(INPUT_PATH, dtype=str, encoding="utf-8")

def clean_text(t):
    t = str(t).replace("\u200b", "")
    return re.sub(r"\s+", " ", t).strip()

n0 = len(df)
d = df[df["comment_text"].notna() & df["final_label"].notna()].copy()
d["comment_text"] = d["comment_text"].map(clean_text)
d = d[d["comment_text"] != ""]
d = d[~d["final_label"].isin(DROP_LABELS)]
n1 = len(d)
d = d.drop_duplicates(subset=["comment_text"], keep="first").reset_index(drop=True)
print(f"Awal {n0} -> setelah buang kosong/{DROP_LABELS} {n1} -> setelah buang duplikat persis {len(d)}")
print(d["final_label"].value_counts())

Awal 1959 -> setelah buang kosong/['netral'] 1875 -> setelah buang duplikat persis 1835
final_label
negatif    1446
positif     389
Name: count, dtype: int64


In [11]:
# ===== EMBEDDING via OLLAMA (di-cache ke .npy, jadi run ulang tidak embed lagi) =====
def embed_texts(texts, batch=32):
    out = []
    for i in range(0, len(texts), batch):
        r = requests.post(f"{OLLAMA_URL}/api/embed",
                          json={"model": EMBED_MODEL, "input": texts[i:i+batch]}, timeout=300)
        r.raise_for_status()
        out.extend(r.json()["embeddings"])
        print(f"\r  embedded {min(i+batch, len(texts))}/{len(texts)}", end="")
    print()
    return np.array(out, dtype=np.float32)

emb_path = OUTPUT_DIR / f"embeddings_{EMBED_MODEL.replace(':','_')}.npy"
txt_path = OUTPUT_DIR / "embeddings_texts.csv"

if emb_path.exists() and txt_path.exists() and \
   pd.read_csv(txt_path, encoding="utf-8")["comment_text"].tolist() == d["comment_text"].tolist():
    emb = np.load(emb_path)
    print("Pakai cache embedding:", emb.shape)
else:
    print("Embedding", len(d), "komentar dengan", EMBED_MODEL, "...")
    emb = embed_texts(d["comment_text"].tolist())
    np.save(emb_path, emb)
    d[["comment_text"]].to_csv(txt_path, index=False, encoding="utf-8")
    print("Embedding selesai:", emb.shape)

emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)   # L2 normalize -> cosine = dot product

Pakai cache embedding: (1835, 1024)


In [12]:
# ===== SWEEP THRESHOLD (per label) - bantu pilih SIM_THRESHOLD =====
def cluster_ids(E, sim_thr):
    if len(E) == 1:
        return np.array([0])
    m = AgglomerativeClustering(n_clusters=None, distance_threshold=1 - sim_thr,
                                metric="cosine", linkage="average")
    return m.fit_predict(E)

rows = []
for lab in sorted(d["final_label"].unique()):
    E = emb[(d["final_label"] == lab).values]
    for thr in [0.70, 0.75, 0.80, 0.85, 0.90]:
        c = cluster_ids(E, thr)
        sizes = np.bincount(c)
        rows.append({"label": lab, "threshold": thr, "n_komentar": len(E), "n_cluster": len(sizes),
                     "singleton": int((sizes == 1).sum()), "cluster_terbesar": int(sizes.max()),
                     "median_ukuran": float(np.median(sizes))})
print(pd.DataFrame(rows).to_string(index=False))

  label  threshold  n_komentar  n_cluster  singleton  cluster_terbesar  median_ukuran
negatif       0.70        1446        884        662               120            1.0
negatif       0.75        1446       1103        949                78            1.0
negatif       0.80        1446       1236       1137                51            1.0
negatif       0.85        1446       1323       1263                34            1.0
negatif       0.90        1446       1377       1346                18            1.0
positif       0.70         389        234        170                39            1.0
positif       0.75         389        304        256                19            1.0
positif       0.80         389        338        313                18            1.0
positif       0.85         389        360        345                 8            1.0
positif       0.90         389        369        354                 6            1.0


In [13]:
# ===== CLUSTER FINAL (per label) dengan SIM_THRESHOLD =====
d["cluster_id"] = ""
for lab in sorted(d["final_label"].unique()):
    idx = np.where((d["final_label"] == lab).values)[0]
    c = cluster_ids(emb[idx], SIM_THRESHOLD)
    d.loc[idx, "cluster_id"] = [f"{lab}_{x:04d}" for x in c]

d["cluster_size"] = d.groupby("cluster_id")["comment_text"].transform("size")
d["panjang"] = d["comment_text"].str.len()
print("Jumlah cluster per label:")
print(d.groupby("final_label")["cluster_id"].nunique())

Jumlah cluster per label:
final_label
negatif    884
positif    234
Name: cluster_id, dtype: int64


In [14]:
# ===== LIHAT ISI CLUSTER TERBESAR (cek apakah memang 1 konteks) =====
LABEL_LIHAT, TOP_N, CONTOH = "negatif", 12, 5

big = (d[d["final_label"] == LABEL_LIHAT].groupby("cluster_id").size()
       .sort_values(ascending=False).head(TOP_N))
for cid, size in big.items():
    sub = d[d["cluster_id"] == cid].sort_values("panjang", ascending=False)
    print(f"\n=== {cid} | {size} komentar ===")
    for t in sub["comment_text"].head(CONTOH):
        print("  -", t[:140])


=== negatif_0107 | 120 komentar ===
  - Saya mendapat laporan saya mendapat laporan saya mendapat laporan saya mendaat laporan saya mendapat laporan
  - saya dapat laporan saya dapat laporan saya dapat laporan. yg ngasih laporan palsu siapa sih
  - saya dapat laporan saya dapat laporan laporan nya selalu bagus bagus dan bagus
  - laporan laporan laporan laporan laporan, s'kli" saya turun lngsung gtu kek 😭
  - saya dapat laporan saya akan saya ingin ujung" nya MBG jg yg dia omongin🤣

=== negatif_0073 | 14 komentar ===
  - nyuklir ..kata ny bukan nuklir gaaesss😁..😱
  - Nuklir nya Nyuklir banget[yummy]
  - bukan NYUKKLIR tapi nukklir
  - NUKLIR BUKAN NYUKLIR🤣🤣🤣
  - Nuklir pke di omongin 😂

=== negatif_0053 | 10 komentar ===
  - "SAYA DAPET LAPORAN" kapan bilang " SAYA LIHAT DAN MENINJAU LANGSUNG "
  - gak ketinggalan kalimat saktinya.... "Saya Dapat Laporan......."
  - klo ditranslate artinya “sodara sodara saya dapat laporan”.
  - coba tebak berapa kali dia ngomong ( SAYA DAPAT LAPORAN 

In [15]:
# ===== (OPSIONAL) LLM OLLAMA KASIH NAMA CLUSTER =====
NAMA_CLUSTER = False          # ganti True kalau mau
LLM_MODEL = "qwen2.5:7b"
NAMA_TOP_N = 12

def ask_llm(prompt):
    r = requests.post(f"{OLLAMA_URL}/api/generate",
                      json={"model": LLM_MODEL, "prompt": prompt, "stream": False,
                            "options": {"temperature": 0.2}}, timeout=300)
    r.raise_for_status()
    return r.json()["response"].strip()

if NAMA_CLUSTER:
    for cid in big.head(NAMA_TOP_N).index:
        contoh = d[d["cluster_id"] == cid]["comment_text"].head(8).tolist()
        p = ("Berikut komentar TikTok tentang Prabowo. Beri SATU nama topik/konteks singkat "
             "(maks 8 kata, bahasa Indonesia) yang menyatukan komentar-komentar ini, dan tulis "
             "apakah semuanya memang satu konteks (ya/tidak).\n\n" + "\n".join(f"- {c}" for c in contoh))
        print(f"\n{cid}: {ask_llm(p)}")

In [16]:
# ===== SIMPAN =====
out_path = OUTPUT_DIR / "clustered.csv"
d.to_csv(out_path, index=False, encoding="utf-8")
print("Tersimpan:", out_path, f"({len(d)} baris)")
print("Urutan baris sama dengan", emb_path.name, "-> bisa dipakai lagi di notebook berikutnya")

Tersimpan: C:\Users\nesha\VSCODE_ALL\PBAAnalysisSentiment\data\praprocessing\labels_shafwa_hans_agreement\clustered.csv (1835 baris)
Urutan baris sama dengan embeddings_bge-m3.npy -> bisa dipakai lagi di notebook berikutnya
